# Part 2h: Weights & Biases Integration for Experiment Tracking

## Overview

In this notebook, we'll explore how to integrate **Weights & Biases (W&B)** for experiment tracking in deep learning projects. W&B is a powerful MLOps platform that helps you track experiments, visualize results, and collaborate with your team.

### Topics Covered:
1. **Setting up W&B** - Installation and initialization
2. **Logging Metrics** - Training metrics, hyperparameters, and artifacts
3. **TensorFlow Integration** - Using W&B callback
4. **PyTorch Integration** - Manual logging in training loops
5. **Hyperparameter Sweeps** - Automated hyperparameter tuning
6. **Model Versioning** - Saving and loading model artifacts

### Why Weights & Biases?
- Track all experiments in one place
- Compare runs visually
- Reproduce experiments easily
- Collaborate with team members
- Version control for models and datasets

In [1]:
# Install required packages (for Google Colab)
!pip install -q tensorflow torch wandb matplotlib seaborn

In [2]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime

# TensorFlow/Keras imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# PyTorch imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# Weights & Biases
import wandb
# Use new Keras 3.x compatible callbacks
try:
    from wandb.integration.keras import WandbMetricsLogger, WandbModelCheckpoint
    WANDB_CALLBACK_TYPE = "new"
except ImportError:
    # Fallback for older wandb versions
    from wandb.integration.keras import WandbCallback
    WANDB_CALLBACK_TYPE = "legacy"

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
torch.manual_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"W&B version: {wandb.__version__}")
print(f"Using W&B callback type: {WANDB_CALLBACK_TYPE}")

TensorFlow version: 2.19.0
PyTorch version: 2.10.0+cu128
W&B version: 0.25.1
Using W&B callback type: new


---

## 1. Setting Up Weights & Biases

### First Time Setup

1. Create an account at [wandb.ai](https://wandb.ai)
2. Get your API key from settings
3. Login using `wandb login`

In [3]:
# Login to W&B (uncomment and run this cell the first time)
# wandb.login()

# Or set API key directly (not recommended for shared notebooks)
# os.environ['WANDB_API_KEY'] = 'your-api-key'

# For this demo, we'll use offline mode (no account needed)
os.environ['WANDB_MODE'] = 'offline'
print("W&B set to offline mode for demo purposes.")
print("To use online mode, comment out the WANDB_MODE line and run wandb.login()")

W&B set to offline mode for demo purposes.
To use online mode, comment out the WANDB_MODE line and run wandb.login()


### W&B Concepts

| Concept | Description |
|---------|-------------|
| **Project** | Collection of related runs |
| **Run** | Single experiment execution |
| **Config** | Hyperparameters and settings |
| **Metrics** | Values logged during training |
| **Artifacts** | Files (models, datasets, etc.) |
| **Sweep** | Hyperparameter search |

---

## 2. Basic W&B Usage

In [4]:
# Initialize a simple W&B run
run = wandb.init(
    project="deep-learning-demo",
    name="basic-example",
    config={
        "learning_rate": 0.001,
        "epochs": 10,
        "batch_size": 64,
        "architecture": "CNN",
        "optimizer": "Adam"
    },
    tags=["demo", "basic"]
)

print(f"Run ID: {run.id}")
print(f"Run Name: {run.name}")
print(f"Project: {run.project}")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


Run ID: 0pf11865
Run Name: basic-example
Project: deep-learning-demo


In [5]:
# Log some sample metrics
for epoch in range(10):
    # Simulate training metrics
    train_loss = 1.0 / (epoch + 1) + np.random.random() * 0.1
    train_acc = 0.5 + epoch * 0.05 + np.random.random() * 0.02
    val_loss = train_loss + 0.1
    val_acc = train_acc - 0.05

    # Log to W&B
    wandb.log({
        "epoch": epoch,
        "train/loss": train_loss,
        "train/accuracy": train_acc,
        "val/loss": val_loss,
        "val/accuracy": val_acc,
        "learning_rate": 0.001 * (0.95 ** epoch)  # Simulated LR decay
    })

print("Sample metrics logged successfully!")

Sample metrics logged successfully!


In [6]:
# Log a matplotlib figure
fig, ax = plt.subplots(figsize=(8, 5))
x = np.linspace(0, 10, 100)
ax.plot(x, np.sin(x), label='sin(x)')
ax.plot(x, np.cos(x), label='cos(x)')
ax.legend()
ax.set_title('Sample Plot')

# Log figure to W&B
wandb.log({"sample_plot": wandb.Image(fig)})
plt.close(fig)

print("Figure logged to W&B!")

Figure logged to W&B!


In [7]:
# End the run
wandb.finish()
print("Run finished!")

epoch,▁▂▃▃▄▅▆▆▇█
learning_rate,█▇▆▅▄▄▃▂▂▁
train/accuracy,▁▂▂▃▄▅▆▆▇█
train/loss,█▄▃▂▂▁▂▁▁▁
val/accuracy,▁▂▂▃▄▅▆▆▇█
val/loss,█▄▃▂▂▁▂▁▁▁
epoch,9
learning_rate,0.00063
train/accuracy,0.95582
train/loss,0.14319
val/accuracy,0.90582


Run finished!


---

## 3. TensorFlow/Keras Integration

W&B provides a convenient callback for Keras that automatically logs metrics, gradients, and model architecture.

In [8]:
# Load Fashion MNIST
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Preprocess
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

# Use subset for faster training
X_train = X_train[:10000]
y_train = y_train[:10000]

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Training data: {X_train.shape}")
print(f"Test data: {X_test.shape}")

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training data: (10000, 28, 28, 1)
Test data: (10000, 28, 28, 1)


In [9]:
def create_cnn_model(learning_rate=0.001, dropout_rate=0.3):
    """Create a CNN model with configurable hyperparameters."""
    model = keras.Sequential([
        layers.Conv2D(32, 3, activation='relu', padding='same', input_shape=(28, 28, 1)),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.GlobalAveragePooling2D(),
        layers.Dropout(dropout_rate),
        layers.Dense(128, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [10]:
# Training with W&B callback
print("Training TensorFlow model with W&B integration:")
print("=" * 60)

# Initialize W&B run
config = {
    "learning_rate": 0.001,
    "epochs": 10,
    "batch_size": 64,
    "dropout_rate": 0.3,
    "architecture": "CNN",
    "dataset": "Fashion-MNIST",
    "optimizer": "Adam"
}

run = wandb.init(
    project="fashion-mnist-tf",
    name=f"cnn-{datetime.now().strftime('%Y%m%d-%H%M%S')}",
    config=config,
    tags=["tensorflow", "cnn", "fashion-mnist"]
)

# Create model
model_tf = create_cnn_model(
    learning_rate=wandb.config.learning_rate,
    dropout_rate=wandb.config.dropout_rate
)

# Use new Keras 3.x compatible callbacks
callbacks = [
    WandbMetricsLogger(log_freq='epoch'),  # Logs metrics each epoch
]

# Train with W&B callbacks
history = model_tf.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=wandb.config.epochs,
    batch_size=wandb.config.batch_size,
    callbacks=callbacks,
    verbose=1
)

Training TensorFlow model with W&B integration:


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 13s 36ms/step - accuracy: 0.5880 - loss: 1.1147 - val_accuracy: 0.0997 - val_loss: 4.6495
Epoch 2/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7630 - loss: 0.6473 - val_accuracy: 0.1000 - val_loss: 6.6038
Epoch 3/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8064 - loss: 0.5312 - val_accuracy: 0.1005 - val_loss: 5.4674
Epoch 4/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8386 - loss: 0.4491 - val_accuracy: 0.7489 - val_loss: 0.6857
Epoch 5/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8536 - loss: 0.4070 - val_accuracy: 0.8251 - val_loss: 0.4807
Epoch 6/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8707 - loss: 0.3653 - val_accuracy: 0.8484 - val_loss: 0.4244
Epoch 7/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8801 - loss: 0.3381 - val_accuracy: 0.8639 - val_loss: 0.3881
Epoch 8/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8889 - loss: 0.3101 - val_accuracy: 

In [11]:
# Log confusion matrix
from sklearn.metrics import confusion_matrix

# Get predictions
y_pred = model_tf.predict(X_test).argmax(axis=1)

# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix')

# Log to W&B
wandb.log({"confusion_matrix": wandb.Image(fig)})
plt.close(fig)

print("Confusion matrix logged!")

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Confusion matrix logged!


In [12]:
# Log sample predictions
sample_images = X_test[:16]
sample_labels = y_test[:16]
sample_preds = model_tf.predict(sample_images).argmax(axis=1)

# Create prediction table
columns = ["image", "true_label", "predicted_label", "correct"]
data = []

for i in range(16):
    data.append([
        wandb.Image(sample_images[i].squeeze()),
        class_names[sample_labels[i]],
        class_names[sample_preds[i]],
        sample_labels[i] == sample_preds[i]
    ])

wandb.log({"predictions": wandb.Table(columns=columns, data=data)})
print("Sample predictions logged!")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Sample predictions logged!


In [13]:
# Finish the TensorFlow run
wandb.finish()
print("TensorFlow run completed!")

epoch/accuracy,▁▅▆▇▇▇████
epoch/epoch,▁▂▃▃▄▅▆▆▇█
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▄▃▂▂▂▂▁▁▁
epoch/val_accuracy,▁▁▁▇██████
epoch/val_loss,▆█▇▁▁▁▁▁▁▁
epoch/accuracy,0.8998
epoch/epoch,9
epoch/learning_rate,0.001
epoch/loss,0.27319
epoch/val_accuracy,0.8656


TensorFlow run completed!


---

## 4. PyTorch Integration

For PyTorch, we manually integrate W&B logging into our training loop.

In [14]:
# Prepare PyTorch data
X_train_pt = torch.tensor(X_train.transpose(0, 3, 1, 2), dtype=torch.float32)
y_train_pt = torch.tensor(y_train, dtype=torch.long)
X_test_pt = torch.tensor(X_test.transpose(0, 3, 1, 2), dtype=torch.float32)
y_test_pt = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_pt, y_train_pt)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

test_dataset = TensorDataset(X_test_pt, y_test_pt)
test_loader = DataLoader(test_dataset, batch_size=64)

print(f"Training batches: {len(train_loader)}")

Training batches: 157


In [15]:
# PyTorch CNN model
class CNNModelPyTorch(nn.Module):
    def __init__(self, dropout_rate=0.3):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2)

        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2)

        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.gap = nn.AdaptiveAvgPool2d(1)

        self.dropout = nn.Dropout(dropout_rate)
        self.fc1 = nn.Linear(128, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.conv3(x))
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

In [16]:
def train_with_wandb(model, train_loader, test_loader, config):
    """
    PyTorch training function with W&B integration.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    # Watch model (logs gradients and parameters)
    wandb.watch(model, criterion, log='all', log_freq=100)

    for epoch in range(config['epochs']):
        # Training
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * data.size(0)
            pred = output.argmax(dim=1)
            train_correct += (pred == target).sum().item()
            train_total += data.size(0)

            # Log batch metrics
            if batch_idx % 50 == 0:
                wandb.log({
                    "batch_loss": loss.item(),
                    "batch": epoch * len(train_loader) + batch_idx
                })

        train_loss /= train_total
        train_acc = train_correct / train_total

        # Validation
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = criterion(output, target)

                val_loss += loss.item() * data.size(0)
                pred = output.argmax(dim=1)
                val_correct += (pred == target).sum().item()
                val_total += data.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total

        # Log epoch metrics
        wandb.log({
            "epoch": epoch,
            "train/loss": train_loss,
            "train/accuracy": train_acc,
            "val/loss": val_loss,
            "val/accuracy": val_acc,
            "learning_rate": optimizer.param_groups[0]['lr']
        })

        scheduler.step()

        print(f"Epoch {epoch+1}/{config['epochs']}: "
              f"Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, "
              f"Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}")

    return model

In [17]:
# Training with W&B
print("Training PyTorch model with W&B integration:")
print("=" * 60)

# Configuration
config = {
    "learning_rate": 0.001,
    "epochs": 10,
    "batch_size": 64,
    "dropout_rate": 0.3,
    "architecture": "CNN",
    "optimizer": "Adam"
}

# Initialize W&B
run = wandb.init(
    project="fashion-mnist-pytorch",
    name=f"cnn-{datetime.now().strftime('%Y%m%d-%H%M%S')}",
    config=config,
    tags=["pytorch", "cnn", "fashion-mnist"]
)

# Create and train model
model_pt = CNNModelPyTorch(dropout_rate=config['dropout_rate'])
trained_model = train_with_wandb(model_pt, train_loader, test_loader, config)

Training PyTorch model with W&B integration:


Epoch 1/10: Train Loss=1.3571, Train Acc=0.4781, Val Loss=0.9727, Val Acc=0.6341
Epoch 2/10: Train Loss=0.9113, Train Acc=0.6577, Val Loss=0.7535, Val Acc=0.7171
Epoch 3/10: Train Loss=0.7765, Train Acc=0.7057, Val Loss=0.6924, Val Acc=0.7376
Epoch 4/10: Train Loss=0.7044, Train Acc=0.7362, Val Loss=0.6631, Val Acc=0.7365
Epoch 5/10: Train Loss=0.6641, Train Acc=0.7524, Val Loss=0.6256, Val Acc=0.7583
Epoch 6/10: Train Loss=0.6024, Train Acc=0.7736, Val Loss=0.5592, Val Acc=0.7838
Epoch 7/10: Train Loss=0.5784, Train Acc=0.7847, Val Loss=0.5450, Val Acc=0.7935
Epoch 8/10: Train Loss=0.5468, Train Acc=0.7951, Val Loss=0.5315, Val Acc=0.7917
Epoch 9/10: Train Loss=0.5321, Train Acc=0.8037, Val Loss=0.5088, Val Acc=0.8096
Epoch 10/10: Train Loss=0.5224, Train Acc=0.8061, Val Loss=0.4830, Val Acc=0.8223


In [18]:
# Save model artifact
model_artifact = wandb.Artifact(
    name="cnn-model",
    type="model",
    description="Fashion MNIST CNN model",
    metadata=config
)

# Save PyTorch model
torch.save(trained_model.state_dict(), "model.pth")
model_artifact.add_file("model.pth")
wandb.log_artifact(model_artifact)

print("Model artifact saved!")

Model artifact saved!


In [19]:
# Finish PyTorch run
wandb.finish()
print("PyTorch run completed!")

batch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
batch_loss,█▄▄▃▄▄▃▂▃▃▂▂▂▂▃▂▃▂▃▃▁▂▂▂▁▂▂▁▁▂▂▁▁▂▂▂▁▁▁▁
epoch,▁▂▃▃▄▅▆▆▇█
learning_rate,█████▁▁▁▁▁
train/accuracy,▁▅▆▇▇▇████
train/loss,█▄▃▃▂▂▁▁▁▁
val/accuracy,▁▄▅▅▆▇▇▇██
val/loss,█▅▄▄▃▂▂▂▁▁
batch,1563
batch_loss,0.48765
epoch,9


PyTorch run completed!


---

## 5. Hyperparameter Sweeps

W&B Sweeps allow automated hyperparameter optimization.

In [20]:
# Define sweep configuration
sweep_config = {
    'method': 'bayes',  # Options: grid, random, bayes
    'metric': {
        'name': 'val/accuracy',
        'goal': 'maximize'
    },
    'parameters': {
        'learning_rate': {
            'distribution': 'log_uniform_values',
            'min': 0.0001,
            'max': 0.01
        },
        'dropout_rate': {
            'values': [0.1, 0.2, 0.3, 0.4, 0.5]
        },
        'batch_size': {
            'values': [32, 64, 128]
        },
        'optimizer': {
            'values': ['adam', 'sgd', 'rmsprop']
        }
    },
    'early_terminate': {
        'type': 'hyperband',
        'min_iter': 3,
        's': 2
    }
}

print("Sweep configuration:")
print(sweep_config)

Sweep configuration:
{'method': 'bayes', 'metric': {'name': 'val/accuracy', 'goal': 'maximize'}, 'parameters': {'learning_rate': {'distribution': 'log_uniform_values', 'min': 0.0001, 'max': 0.01}, 'dropout_rate': {'values': [0.1, 0.2, 0.3, 0.4, 0.5]}, 'batch_size': {'values': [32, 64, 128]}, 'optimizer': {'values': ['adam', 'sgd', 'rmsprop']}}, 'early_terminate': {'type': 'hyperband', 'min_iter': 3, 's': 2}}


In [21]:
def sweep_train():
    """
    Training function for sweep agent.

    W&B will call this function with different hyperparameter combinations.
    """
    # Initialize run (config is automatically set by sweep agent)
    run = wandb.init()
    config = wandb.config

    # Create model with sweep config
    model = create_cnn_model(
        learning_rate=config.learning_rate,
        dropout_rate=config.dropout_rate
    )

    # Train for a few epochs (abbreviated for sweep)
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=5,
        batch_size=config.batch_size,
        callbacks=[WandbMetricsLogger(log_freq='epoch')],
        verbose=0
    )

    # wandb.finish() is called automatically

print("Sweep training function defined.")
print("To run a sweep:")
print("1. sweep_id = wandb.sweep(sweep_config, project='my-project')")
print("2. wandb.agent(sweep_id, sweep_train, count=10)")

Sweep training function defined.
To run a sweep:
1. sweep_id = wandb.sweep(sweep_config, project='my-project')
2. wandb.agent(sweep_id, sweep_train, count=10)


In [22]:
# Demo: Run a mini sweep (just 2 runs for demonstration)
print("Running mini sweep demonstration:")
print("=" * 60)

# Manual mini-sweep (simulating what W&B agent does)
sweep_configs = [
    {'learning_rate': 0.001, 'dropout_rate': 0.3, 'batch_size': 64},
    {'learning_rate': 0.0005, 'dropout_rate': 0.2, 'batch_size': 32}
]

sweep_results = []

for i, cfg in enumerate(sweep_configs):
    print(f"\nSweep run {i+1}/{len(sweep_configs)}")
    print(f"Config: {cfg}")

    run = wandb.init(
        project="fashion-mnist-sweep-demo",
        name=f"sweep-run-{i+1}",
        config=cfg,
        reinit=True
    )

    model = create_cnn_model(
        learning_rate=cfg['learning_rate'],
        dropout_rate=cfg['dropout_rate']
    )

    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=3,
        batch_size=cfg['batch_size'],
        callbacks=[WandbMetricsLogger(log_freq='epoch')],
        verbose=1
    )

    final_val_acc = history.history['val_accuracy'][-1]
    sweep_results.append({'config': cfg, 'val_accuracy': final_val_acc})

    wandb.finish()

# Print sweep results
print("\n" + "="*60)
print("Sweep Results:")
for result in sweep_results:
    print(f"Config: {result['config']} -> Val Accuracy: {result['val_accuracy']:.4f}")

Running mini sweep demonstration:

Sweep run 1/2
Config: {'learning_rate': 0.001, 'dropout_rate': 0.3, 'batch_size': 64}


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Epoch 1/3


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


157/157 ━━━━━━━━━━━━━━━━━━━━ 10s 31ms/step - accuracy: 0.5740 - loss: 1.1471 - val_accuracy: 0.1000 - val_loss: 3.4508
Epoch 2/3
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7443 - loss: 0.6901 - val_accuracy: 0.1248 - val_loss: 4.0713
Epoch 3/3
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7944 - loss: 0.5607 - val_accuracy: 0.5011 - val_loss: 1.6231


epoch/accuracy,▁▆█
epoch/epoch,▁▅█
epoch/learning_rate,▁▁▁
epoch/loss,█▃▁
epoch/val_accuracy,▁▁█
epoch/val_loss,▆█▁
epoch/accuracy,0.7944
epoch/epoch,2
epoch/learning_rate,0.001
epoch/loss,0.56072
epoch/val_accuracy,0.5011



Sweep run 2/2
Config: {'learning_rate': 0.0005, 'dropout_rate': 0.2, 'batch_size': 32}


Epoch 1/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 18ms/step - accuracy: 0.6183 - loss: 1.0420 - val_accuracy: 0.1097 - val_loss: 2.3570
Epoch 2/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.7637 - loss: 0.6407 - val_accuracy: 0.7247 - val_loss: 0.6864
Epoch 3/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8145 - loss: 0.5126 - val_accuracy: 0.8414 - val_loss: 0.4530


epoch/accuracy,▁▆█
epoch/epoch,▁▅█
epoch/learning_rate,▁▁▁
epoch/loss,█▃▁
epoch/val_accuracy,▁▇█
epoch/val_loss,█▂▁
epoch/accuracy,0.8145
epoch/epoch,2
epoch/learning_rate,0.0005
epoch/loss,0.51257
epoch/val_accuracy,0.8414



Sweep Results:
Config: {'learning_rate': 0.001, 'dropout_rate': 0.3, 'batch_size': 64} -> Val Accuracy: 0.5011
Config: {'learning_rate': 0.0005, 'dropout_rate': 0.2, 'batch_size': 32} -> Val Accuracy: 0.8414


---

## 6. Custom Logging and Visualizations

In [23]:
# Custom W&B logging examples
print("Custom W&B Logging Examples:")
print("=" * 60)

run = wandb.init(
    project="custom-logging-demo",
    name="logging-examples"
)

# 1. Log histograms
weights = np.random.randn(1000)
wandb.log({"weight_distribution": wandb.Histogram(weights)})
print("Histogram logged!")

# 2. Log images with captions
sample_image = X_train[0].squeeze()
wandb.log({
    "sample_image": wandb.Image(
        sample_image,
        caption=f"Label: {class_names[y_train[0]]}"
    )
})
print("Image with caption logged!")

# 3. Log multiple images in a grid
images = [wandb.Image(X_train[i].squeeze(), caption=class_names[y_train[i]])
          for i in range(9)]
wandb.log({"image_grid": images})
print("Image grid logged!")

# 4. Log HTML content
html_content = """
<h3>Training Summary</h3>
<table>
    <tr><td>Model</td><td>CNN</td></tr>
    <tr><td>Dataset</td><td>Fashion MNIST</td></tr>
    <tr><td>Best Accuracy</td><td>92.5%</td></tr>
</table>
"""
wandb.log({"summary_html": wandb.Html(html_content)})
print("HTML content logged!")

# 5. Log a summary metric
wandb.summary["best_model_accuracy"] = 0.925
wandb.summary["total_parameters"] = sum(p.numel() for p in trained_model.parameters())
print("Summary metrics logged!")

# 6. Log custom chart data
data = [[x, np.sin(x)] for x in np.linspace(0, 2*np.pi, 100)]
table = wandb.Table(data=data, columns=["x", "sin(x)"])
wandb.log({"sine_wave": wandb.plot.line(table, "x", "sin(x)", title="Sine Wave")})
print("Custom chart logged!")

wandb.finish()
print("\nAll custom logging examples completed!")

Custom W&B Logging Examples:


Histogram logged!
Image with caption logged!
Image grid logged!
HTML content logged!
Summary metrics logged!
Custom chart logged!


best_model_accuracy,0.925
total_parameters,110666



All custom logging examples completed!


---

## 7. Summary and Best Practices

In [24]:
import pandas as pd

# Summary table
summary_data = {
    'Feature': ['wandb.init()', 'wandb.log()', 'WandbCallback', 'wandb.watch()',
                'wandb.Artifact', 'wandb.sweep()', 'wandb.Table'],
    'Purpose': ['Initialize run', 'Log metrics', 'Keras auto-logging', 'Log PyTorch gradients',
                'Version models/data', 'Hyperparameter search', 'Log structured data'],
    'Framework': ['Both', 'Both', 'TensorFlow', 'PyTorch', 'Both', 'Both', 'Both'],
    'Common Args': ['project, name, config', 'dict of metrics', 'save_model, log_gradients',
                    'model, criterion', 'name, type', 'sweep_config', 'data, columns']
}

df = pd.DataFrame(summary_data)
print("W&B Features Summary:")
print("=" * 100)
print(df.to_string(index=False))

W&B Features Summary:
       Feature               Purpose  Framework               Common Args
  wandb.init()        Initialize run       Both     project, name, config
   wandb.log()           Log metrics       Both           dict of metrics
 WandbCallback    Keras auto-logging TensorFlow save_model, log_gradients
 wandb.watch() Log PyTorch gradients    PyTorch          model, criterion
wandb.Artifact   Version models/data       Both                name, type
 wandb.sweep() Hyperparameter search       Both              sweep_config
   wandb.Table   Log structured data       Both             data, columns


### Best Practices for W&B Integration

1. **Consistent Project Naming**
   - Use descriptive project names
   - Group related experiments together

2. **Comprehensive Config Logging**
   - Log all hyperparameters
   - Include environment info (GPU, library versions)

3. **Meaningful Run Names**
   - Use timestamps or descriptive names
   - Add tags for easy filtering

4. **Regular Metric Logging**
   - Log at consistent intervals
   - Include both train and validation metrics

5. **Model Artifacts**
   - Save best models as artifacts
   - Include metadata about training

6. **Use Sweeps for Hyperparameter Tuning**
   - Define search space thoughtfully
   - Use early termination for efficiency

In [25]:
# Template for W&B integration
template = '''
# W&B Integration Template

import wandb

# 1. Initialize
wandb.init(
    project="my-project",
    name="experiment-name",
    config={
        "learning_rate": 0.001,
        "epochs": 10,
        "batch_size": 64
    },
    tags=["experiment", "v1"]
)

# 2. Training loop
for epoch in range(wandb.config.epochs):
    # Train...
    train_loss, train_acc = train_epoch(...)
    val_loss, val_acc = evaluate(...)

    # Log metrics
    wandb.log({
        "epoch": epoch,
        "train/loss": train_loss,
        "train/accuracy": train_acc,
        "val/loss": val_loss,
        "val/accuracy": val_acc
    })

# 3. Save model
artifact = wandb.Artifact("model", type="model")
artifact.add_file("model.pt")
wandb.log_artifact(artifact)

# 4. Finish
wandb.finish()
'''

print("W&B Integration Template:")
print("=" * 60)
print(template)

W&B Integration Template:

# W&B Integration Template

import wandb

# 1. Initialize
wandb.init(
    project="my-project",
    name="experiment-name",
    config={
        "learning_rate": 0.001,
        "epochs": 10,
        "batch_size": 64
    },
    tags=["experiment", "v1"]
)

# 2. Training loop
for epoch in range(wandb.config.epochs):
    # Train...
    train_loss, train_acc = train_epoch(...)
    val_loss, val_acc = evaluate(...)
    
    # Log metrics
    wandb.log({
        "epoch": epoch,
        "train/loss": train_loss,
        "train/accuracy": train_acc,
        "val/loss": val_loss,
        "val/accuracy": val_acc
    })

# 3. Save model
artifact = wandb.Artifact("model", type="model")
artifact.add_file("model.pt")
wandb.log_artifact(artifact)

# 4. Finish
wandb.finish()



In [26]:
# Clean up
if os.path.exists("model.pth"):
    os.remove("model.pth")

print("\nNotebook completed successfully!")
print("="*50)
print("Key takeaways:")
print("1. W&B provides comprehensive experiment tracking")
print("2. TensorFlow uses WandbCallback for automatic logging")
print("3. PyTorch requires manual wandb.log() calls")
print("4. Sweeps automate hyperparameter optimization")
print("5. Artifacts enable model versioning and sharing")


Notebook completed successfully!
Key takeaways:
1. W&B provides comprehensive experiment tracking
2. TensorFlow uses WandbCallback for automatic logging
3. PyTorch requires manual wandb.log() calls
4. Sweeps automate hyperparameter optimization
5. Artifacts enable model versioning and sharing
